## RAID and quotas

**RAID** ("Redundant Array of Independent Disks") combines several disks into one logical device for redundancy, speed, or both. Linux does **software RAID** via **`mdadm`**, no hardware controller needed. The levels:

| Level | What | Disks | Survives |
|---|---|---|---|
| **0** stripe | split across disks — fastest | 2+ | **nothing** — one dies, all lost |
| **1** mirror | identical copy on each | 2+ | up to N−1 failures |
| **5** | stripe + 1 parity | 3+ | 1 failure |
| **6** | stripe + 2 parity | 4+ | 2 failures |
| **10** mirror+stripe | best speed + safety | 4+ | 1 per mirror pair |

Rules of thumb: **RAID 1** for simple redundancy (boot drives); **RAID 10** the default for performance + reliability; **RAID 6** over RAID 5 on big modern disks (RAID 5 rebuilds are slow and risky). ⚠️ **RAID is not a backup** — it protects against *disk failure*, not `rm -rf`, ransomware, or corruption, which replicate to every disk. Always keep separate backups.

```bash
sudo mdadm --create /dev/md0 --level=1 --raid-devices=2 /dev/sdb1 /dev/sdc1
cat /proc/mdstat                    # status of all arrays
sudo mdadm --detail /dev/md0        # detail + disk states
```

The array is one block device (`/dev/md0`) you format and mount. Replace a failed disk with `--fail` / `--remove` / `--add`, and watch the rebuild in `/proc/mdstat`.

**Quotas** cap how much a user or group can use on a filesystem. Enable with `usrquota,grpquota` in the fstab options, initialise with `quotacheck -cug` + `quotaon`, set limits with **`edquota -u alice`** (soft = warning + grace, hard = absolute ceiling), and report with **`repquota -a`**. On XFS it's the `uquota` mount option plus `xfs_quota`.
